# EDA - entidad Solicitante

Datos demograficos y financieros estaticos del que pide el prestamo.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

COLS = ['person_age', 'person_gender', 'person_education', 'person_income',
        'person_emp_exp', 'person_home_ownership', 'cb_person_cred_hist_length',
        'credit_score', 'previous_loan_defaults_on_file']

df_full = pd.read_csv(ROOT / 'data' / 'loan_data.csv')
sol = df_full[COLS].copy()
sol['loan_status'] = df_full['loan_status']
sol.shape

## Vista general

In [ ]:
sol.head()

In [ ]:
sol.describe(include='all').T

## Distribuciones categoricas

In [ ]:
for col in sol.select_dtypes(include='object').columns:
    print(f'\n>> {col}')
    print(sol[col].value_counts(normalize=True).round(3))

## Outliers y violaciones de reglas (cap. 9)

In [ ]:
violaciones = {
    'person_age fuera de [18, 100]':       int((~sol['person_age'].between(18, 100)).sum()),
    'credit_score fuera de [300, 850]':    int((~sol['credit_score'].between(300, 850)).sum()),
    'person_income negativo':              int((sol['person_income'] < 0).sum()),
    'person_emp_exp > person_age - 18':    int((sol['person_emp_exp'] > sol['person_age'] - 18).sum()),
    'cred_hist_length > person_age':       int((sol['cb_person_cred_hist_length'] > sol['person_age']).sum()),
}
pd.DataFrame.from_dict(violaciones, orient='index', columns=['n_violaciones'])

In [ ]:
sol['person_income'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2)

## Que variables del solicitante predicen default

In [ ]:
# Numericas
num = sol.select_dtypes(include='number')
num.corr()['loan_status'].drop('loan_status').abs().sort_values(ascending=False).to_frame('|corr|').round(4)

In [ ]:
# Categoricas (spread max-min de tasa de default)
for col in sol.select_dtypes(include='object').columns:
    tabla = sol.groupby(col).agg(
        n=('loan_status', 'size'),
        tasa_default=('loan_status', 'mean'),
    ).round(4).sort_values('tasa_default', ascending=False)
    spread = tabla['tasa_default'].max() - tabla['tasa_default'].min()
    print(f'\n>> {col}  (spread: {spread:.4f})')
    print(tabla)

## Hallazgo

- `previous_loan_defaults_on_file` (spread 0.45) y `person_home_ownership` (spread 0.26) son los predictores reales del solicitante.
- `credit_score` (|corr| 0.008) y `person_age` (|corr| 0.02) **no aportan** señal en este dataset.